# AI-Powered PDF Intelligence & Question Answering Assistant (RAG-Based)

1. Developed an AI-powered PDF Question Answering Assistant using Python, Gradio, and OpenRouter API.
2. Integrated PDF processing with PyPDF to extract text and retrieve relevant information from uploaded documents.
3. Used an LLM (GPT model through OpenRouter) to answer user questions based on PDF content.
4. Implemented conversation history to support follow-up questions and provide contextual responses.
5. Developed an automatic email extraction feature using Regex to identify and save unique email addresses in a local emails.txt file.

In [ ]:
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install gradio openai python-dotenv pypdf

In [18]:
import os
import re
import gradio as gr

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

c:\Users\nadia\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Please check your .env file."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

print("API key loaded successfully.")

API key loaded successfully.


In [20]:
system_prompt = """
You are a helpful PDF Question Answering Assistant.

Instructions:

1. Answer questions using only the uploaded PDF content.
2. Do not invent or assume information.
3. If the answer is not available in the PDF, clearly say:
   "This information is not available in the uploaded PDF."
4. Provide clear and easy-to-understand answers.
5. Include page numbers when possible.
6. Summarize information when requested.
7. Use conversation history to understand follow-up questions.
"""

In [21]:
def read_pdf(pdf_file):

    if pdf_file is None:
        return []

    reader = PdfReader(pdf_file)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text() or ""

        if text.strip():

            pages.append({
                "page": page_number,
                "text": text
            })

    return pages

In [22]:
def save_emails(message):

    emails = re.findall(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
        message
    )

    emails = list(dict.fromkeys(
        email.lower() for email in emails
    ))

    if not emails:
        return []

    file_name = "emails.txt"

    # Read existing email addresses
    existing_emails = set()

    if os.path.exists(file_name):

        with open(file_name, "r", encoding="utf-8") as file:

            existing_emails = {
                line.strip().lower()
                for line in file
                if line.strip()
            }

    # Save only new email addresses
    new_emails = [
        email for email in emails
        if email not in existing_emails
    ]

    if new_emails:

        with open(file_name, "a", encoding="utf-8") as file:

            for email in new_emails:
                file.write(email + "\n")

        print("New email addresses saved:", new_emails)

    return emails

In [23]:
def save_emails(message):

    emails = re.findall(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
        message
    )

    emails = list(dict.fromkeys(
        email.lower() for email in emails
    ))

    if not emails:
        return []

    file_name = "emails.txt"

    # Read existing email addresses
    existing_emails = set()

    if os.path.exists(file_name):

        with open(file_name, "r", encoding="utf-8") as file:

            existing_emails = {
                line.strip().lower()
                for line in file
                if line.strip()
            }

    # Save only new email addresses
    new_emails = [
        email for email in emails
        if email not in existing_emails
    ]

    if new_emails:

        with open(file_name, "a", encoding="utf-8") as file:

            for email in new_emails:
                file.write(email + "\n")

        print("New email addresses saved:", new_emails)

    return emails

In [24]:
def retrieve_relevant_pages(question, pages, top_k=6):

    stop_words = {
        "what", "is", "are", "the", "a", "an",
        "of", "in", "on", "for", "to", "and",
        "how", "why", "when", "where", "who",
        "can", "you", "me", "explain", "tell",
        "about", "please", "summarize"
    }

    question_words = set(
        re.findall(r'\b\w+\b', question.lower())
    )

    question_words = question_words - stop_words

    scores = []

    for page in pages:

        page_words = set(
            re.findall(r'\b\w+\b', page["text"].lower())
        )

        score = len(question_words.intersection(page_words))

        scores.append({
            "page": page["page"],
            "text": page["text"],
            "score": score
        })

    scores.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    relevant_pages = [
        page for page in scores[:top_k]
        if page["score"] > 0
    ]

    if not relevant_pages:
        relevant_pages = pages[:3]

    context = ""

    for page in relevant_pages:

        context += (
            f"\nPage {page['page']}:\n"
            f"{page['text']}\n"
        )

    return context[:16000]

    try:

        # Step 1: Extract and save email addresses
        found_emails = save_emails(message)

        # Step 2: Check PDF upload
        if pdf_file is None:
            return "Please upload a PDF file first."

        # Step 3: Read PDF
        pages = read_pdf(pdf_file)

        if not pages:
            return "No readable text found in this PDF."

        # Step 4: Retrieve relevant PDF content
        pdf_context = retrieve_relevant_pages(
            message,
            pages
        )

        # Step 5: Prepare conversation messages
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "system",
                "content": f"Uploaded PDF content:\n{pdf_context}"
            }
        ] + history + [
            {
                "role": "user",
                "content": message
            }
        ]

        # Step 6: Send request to OpenRouter
        response = client.chat.completions.create(

            model="openai/gpt-5.4-mini",

            messages=messages,

            max_tokens=2000
        )

        # Step 7: Return chatbot response
        return response.choices[0].message.content

    except Exception as e:

        return f"Error: {str(e)}"

In [25]:
def save_emails(message):

    emails = re.findall(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
        message
    )

    emails = list(dict.fromkeys(
        email.lower() for email in emails
    ))

    if not emails:
        return []

    file_name = "emails.txt"

    existing_emails = set()

    if os.path.exists(file_name):

        with open(file_name, "r", encoding="utf-8") as file:

            existing_emails = {
                line.strip().lower()
                for line in file
                if line.strip()
            }

    new_emails = [
        email for email in emails
        if email not in existing_emails
    ]

    if new_emails:

        with open(file_name, "a", encoding="utf-8") as file:

            for email in new_emails:
                file.write(email + "\n")

        print("New email addresses saved:", new_emails)

    return emails

In [26]:
def chat(message, history, pdf_file):

    try:

        # Step 1: Save email addresses
        found_emails = save_emails(message)

        if found_emails:
            print("Emails detected:", found_emails)

        # Step 2: Check PDF upload
        if pdf_file is None:
            return "Please upload a PDF file first."

        # Step 3: Read PDF
        pages = read_pdf(pdf_file)

        if not pages:
            return "No readable text found in this PDF."

        # Step 4: Retrieve relevant PDF content
        pdf_context = retrieve_relevant_pages(
            message,
            pages,
            top_k=6
        )

        # Step 5: Prepare conversation messages
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "system",
                "content": (
                    "Uploaded PDF content:\n"
                    + pdf_context
                )
            }
        ]

        # Step 6: Add previous conversation history
        for item in (history or []):

            if (
                isinstance(item, dict)
                and item.get("role") in ["user", "assistant"]
                and isinstance(item.get("content"), str)
            ):

                messages.append({
                    "role": item["role"],
                    "content": item["content"]
                })

        # Step 7: Add current user message
        messages.append({
            "role": "user",
            "content": message
        })

        # Step 8: Send request to OpenRouter
        response = client.chat.completions.create(

            model="openai/gpt-5.4-mini",

            messages=messages,

            max_tokens=2000
        )

        # Step 9: Return chatbot response
        answer = response.choices[0].message.content

        return answer or "No response received from the model."

    except Exception as e:

        return f"Error: {str(e)}"

In [27]:

demo = gr.ChatInterface(
    fn=chat,

    additional_inputs=[
        gr.File(
            label="Upload your PDF",
            file_types=[".pdf"],
            type="filepath"
        )
    ],

    title="PDF Question Answering Assistant",

    description="""
    Upload a PDF and ask questions about its content.

    Email addresses entered in chat messages are
    automatically saved in emails.txt.
    """,

    examples=[
        ["Summarize this PDF.", None],
        ["What are the main topics discussed?", None],
        ["Explain the key findings.", None],
        ["What are the important points?", None]
    ]
)

demo.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


New email addresses saved: ['a@mail.com', 'b@gmail.com', 'c@gmail.com']
Emails detected: ['a@mail.com', 'b@gmail.com', 'c@gmail.com']
